# Inventory Replenishment Agent

This notebook loads `sales.csv`, `inventory.csv`, and `params.csv`, then:

- aggregates daily demand per SKU
- builds an EWMA forecast per SKU
- computes safety stock from forecast errors and service level
- simulates daily replenishment decisions
- tracks fill rate, stockouts, and cost
- prints a daily agent log with rationale


In [ ]:
import pandas as pd
import numpy as np
from math import ceil, sqrt
from statistics import NormalDist

In [ ]:
sales = pd.read_csv("sales.csv", parse_dates=["date"])
inventory = pd.read_csv("inventory.csv")
params = pd.read_csv("params.csv")

print("sales.csv")
display(sales.head())

print("inventory.csv")
display(inventory)

print("params.csv")
display(params)

sales.csv


,date,sku,qty_sold
0,2026-01-01,SKU_A,19
1,2026-01-02,SKU_A,18
2,2026-01-03,SKU_A,24
3,2026-01-04,SKU_A,23
4,2026-01-05,SKU_A,11


inventory.csv


,sku,opening_stock
0,SKU_A,85
1,SKU_B,60
2,SKU_C,110


params.csv


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU_A,10,0.05,4.0,3,20,0.95
1,SKU_B,14,0.07,6.0,4,15,0.97
2,SKU_C,8,0.04,3.5,2,25,0.94


In [ ]:
sales = pd.read_csv("sales.csv", parse_dates=["date"])
inventory = pd.read_csv("inventory.csv")
params = pd.read_csv("params.csv")

print("sales.csv")
display(sales.head())

print("inventory.csv")
display(inventory)

print("params.csv")
display(params)

sales.csv


,date,sku,qty_sold
0,2026-01-01,SKU_A,19
1,2026-01-02,SKU_A,18
2,2026-01-03,SKU_A,24
3,2026-01-04,SKU_A,23
4,2026-01-05,SKU_A,11


inventory.csv


,sku,opening_stock
0,SKU_A,85
1,SKU_B,60
2,SKU_C,110


params.csv


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU_A,10,0.05,4.0,3,20,0.95
1,SKU_B,14,0.07,6.0,4,15,0.97
2,SKU_C,8,0.04,3.5,2,25,0.94


## Prepare daily demand by SKU

In [ ]:
sales = (
    sales.groupby(["date", "sku"], as_index=False)["qty_sold"]
    .sum()
    .sort_values(["sku", "date"])
)

all_dates = pd.date_range(sales["date"].min(), sales["date"].max(), freq="D")
all_skus = sales["sku"].unique()

full_index = pd.MultiIndex.from_product([all_dates, all_skus], names=["date", "sku"])
sales = (
    sales.set_index(["date", "sku"])
    .reindex(full_index, fill_value=0)
    .reset_index()
    .sort_values(["sku", "date"])
)

sku_inputs = inventory.merge(params, on="sku", how="inner")

display(sales.head(10))
display(sku_inputs)

,date,sku,qty_sold
0,2026-01-01,SKU_A,19
3,2026-01-02,SKU_A,18
6,2026-01-03,SKU_A,24
9,2026-01-04,SKU_A,23
12,2026-01-05,SKU_A,11
15,2026-01-06,SKU_A,10
18,2026-01-07,SKU_A,15
21,2026-01-08,SKU_A,17
24,2026-01-09,SKU_A,21
27,2026-01-10,SKU_A,20


,sku,opening_stock,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU_A,85,10,0.05,4.0,3,20,0.95
1,SKU_B,60,14,0.07,6.0,4,15,0.97
2,SKU_C,110,8,0.04,3.5,2,25,0.94


## Forecast helpers

In [ ]:
def ewma_forecast(series, alpha=0.30):
    forecast = []
    prev = float(series.iloc[0])
    for actual in series:
        forecast.append(prev)
        prev = alpha * float(actual) + (1 - alpha) * prev
    return pd.Series(forecast, index=series.index)

def z_value(service_level):
    return NormalDist().inv_cdf(service_level)

## Simulate replenishment agent

In [ ]:
review_period = 1
fixed_order_cost = 0.0

results = []
daily_logs = []

for _, row in sku_inputs.iterrows():
    sku = row["sku"]
    opening_stock = int(row["opening_stock"])
    unit_cost = float(row["unit_cost"])
    holding_cost_per_day = float(row["holding_cost_per_day"])
    stockout_cost = float(row["stockout_cost"])
    lead_time_days = int(row["lead_time_days"])
    min_order_qty = int(row["min_order_qty"])
    service_level = float(row["service_level"])

    sku_sales = sales[sales["sku"] == sku].copy().sort_values("date").reset_index(drop=True)
    sku_sales["forecast"] = ewma_forecast(sku_sales["qty_sold"], alpha=0.30)

    on_hand = opening_stock
    backorder = 0
    pipeline = []  # each entry: {"arrival_date": pd.Timestamp, "qty": int}

    total_holding_cost = 0.0
    total_stockout_cost = 0.0
    total_order_cost = 0.0
    total_demand = 0
    total_fulfilled = 0
    total_stockout_units = 0

    past_errors = []

    for i, day in sku_sales.iterrows():
        current_date = day["date"]
        demand = int(day["qty_sold"])
        forecast_today = max(0.0, float(day["forecast"]))

        arrivals_today = sum(po["qty"] for po in pipeline if po["arrival_date"] == current_date)
        on_hand += arrivals_today
        pipeline = [po for po in pipeline if po["arrival_date"] != current_date]

        if len(past_errors) >= 2:
            sigma = float(np.std(past_errors, ddof=1))
        else:
            sigma = max(1.0, float(sku_sales["qty_sold"].iloc[: max(2, i + 1)].std(ddof=0) if i > 0 else 1.0))

        z = z_value(service_level)
        protection_period = lead_time_days + review_period
        expected_demand_pp = forecast_today * protection_period
        safety_stock = z * sigma * sqrt(protection_period)
        reorder_point = expected_demand_pp + safety_stock

        pipeline_qty = sum(po["qty"] for po in pipeline)
        inventory_position = on_hand + pipeline_qty - backorder

        action = "WAIT"
        order_qty = 0
        reason = ""

        if inventory_position <= reorder_point:
            target_level = reorder_point
            raw_order = target_level - inventory_position
            order_qty = max(min_order_qty, int(ceil(raw_order)))
            arrival_date = current_date + pd.Timedelta(days=lead_time_days)
            pipeline.append({"arrival_date": arrival_date, "qty": order_qty})
            total_order_cost += fixed_order_cost
            action = "ORDER"
            reason = (
                f"Inventory position {inventory_position:.1f} is at or below reorder point {reorder_point:.1f}. "
                f"Expected demand over protection period = {expected_demand_pp:.1f}, "
                f"safety stock = {safety_stock:.1f}, "
                f"so the agent ordered {order_qty} units."
            )
        else:
            reason = (
                f"Inventory position {inventory_position:.1f} is above reorder point {reorder_point:.1f}. "
                f"The agent waits to avoid unnecessary holding cost."
            )

        total_required = backorder + demand
        fulfilled = min(on_hand, total_required)
        on_hand -= fulfilled
        unmet = total_required - fulfilled
        backorder = unmet

        total_demand += demand
        fulfilled_current = max(0, demand - unmet)
        total_fulfilled += fulfilled_current
        total_stockout_units += unmet

        holding_cost_today = on_hand * holding_cost_per_day
        stockout_cost_today = unmet * stockout_cost

        total_holding_cost += holding_cost_today
        total_stockout_cost += stockout_cost_today

        past_errors.append(demand - forecast_today)

        daily_logs.append({
            "date": current_date.date(),
            "sku": sku,
            "demand": demand,
            "forecast": round(forecast_today, 2),
            "sigma": round(sigma, 2),
            "safety_stock": round(safety_stock, 2),
            "reorder_point": round(reorder_point, 2),
            "arrivals_today": arrivals_today,
            "inventory_position": round(inventory_position, 2),
            "on_hand_end": on_hand,
            "pipeline_qty": sum(po["qty"] for po in pipeline),
            "backorder": backorder,
            "action": action,
            "order_qty": order_qty,
            "holding_cost_today": round(holding_cost_today, 2),
            "stockout_cost_today": round(stockout_cost_today, 2),
            "reason": reason,
        })

    fill_rate = total_fulfilled / total_demand if total_demand > 0 else 1.0
    total_cost = total_holding_cost + total_stockout_cost + total_order_cost

    results.append({
        "sku": sku,
        "total_demand": total_demand,
        "fulfilled_units": total_fulfilled,
        "stockout_units": total_stockout_units,
        "fill_rate": round(fill_rate, 4),
        "holding_cost": round(total_holding_cost, 2),
        "stockout_cost": round(total_stockout_cost, 2),
        "order_cost": round(total_order_cost, 2),
        "total_cost": round(total_cost, 2),
        "ending_on_hand": on_hand,
        "ending_backorder": backorder,
    })

daily_log_df = pd.DataFrame(daily_logs)
summary_df = pd.DataFrame(results)

print("=== SUMMARY ===")
display(summary_df)

print("=== DAILY AGENT LOG (first 25 rows) ===")
display(daily_log_df.head(25))

=== SUMMARY ===


,sku,total_demand,fulfilled_units,stockout_units,fill_rate,holding_cost,stockout_cost,order_cost,total_cost,ending_on_hand,ending_backorder
0,SKU_A,1741,1735,6,0.9966,96.15,24.0,0.0,120.15,11,0
1,SKU_B,1088,1082,6,0.9945,110.53,36.0,0.0,146.53,0,0
2,SKU_C,2097,2062,35,0.9833,88.68,122.5,0.0,211.18,26,0


=== DAILY AGENT LOG (first 25 rows) ===


,date,sku,demand,forecast,sigma,safety_stock,reorder_point,arrivals_today,inventory_position,on_hand_end,pipeline_qty,backorder,action,order_qty,holding_cost_today,stockout_cost_today,reason
0,2026-01-01,SKU_A,19,19.00,1.00,3.29,79.29,0,85,66,0,0,WAIT,0,3.30,0.0,Inventory position 85.0 is above reorder point...
1,2026-01-02,SKU_A,18,19.00,1.00,3.29,79.29,0,66,48,20,0,ORDER,20,2.40,0.0,Inventory position 66.0 is at or below reorder...
2,2026-01-03,SKU_A,24,18.70,0.71,2.33,77.13,0,68,24,40,0,ORDER,20,1.20,0.0,Inventory position 68.0 is at or below reorder...
3,2026-01-04,SKU_A,23,20.29,3.39,11.14,92.30,0,64,1,69,0,ORDER,29,0.05,0.0,Inventory position 64.0 is at or below reorder...
4,2026-01-05,SKU_A,11,21.10,2.84,9.33,93.75,20,70,10,73,0,ORDER,24,0.50,0.0,Inventory position 70.0 is at or below reorder...
5,2026-01-06,SKU_A,10,18.07,5.84,19.22,91.51,20,83,20,73,0,ORDER,20,1.00,0.0,Inventory position 83.0 is at or below reorder...
6,2026-01-07,SKU_A,15,15.65,6.05,19.90,82.50,29,93,34,44,0,WAIT,0,1.70,0.0,Inventory position 93.0 is above reorder point...
7,2026-01-08,SKU_A,17,15.46,5.54,18.22,80.05,24,78,41,40,0,ORDER,20,2.05,0.0,Inventory position 78.0 is at or below reorder...
8,2026-01-09,SKU_A,21,15.92,5.25,17.29,80.96,20,81,40,20,0,WAIT,0,2.00,0.0,Inventory position 81.0 is above reorder point...
9,2026-01-10,SKU_A,20,17.44,5.35,17.61,87.38,0,60,20,48,0,ORDER,28,1.00,0.0,Inventory position 60.0 is at or below reorder...


save outputs

In [ ]:
daily_log_df.to_csv("daily_agent_log.csv", index=False)
summary_df.to_csv("summary_results.csv", index=False)

print("Saved daily_agent_log.csv and summary_results.csv")

Saved daily_agent_log.csv and summary_results.csv
